In [ ]:
from __future__ import annotations

import gc
import logging
import os
import sys
from pathlib import Path
from typing import Dict, Iterator, Optional, Tuple
import contextlib
import matplotlib.pyplot as plt
import json

import numpy as np
import torch
import torch.nn.functional as F
from huggingface_hub import login


# Helper for Python < 3.10 compatibility
class nullcontext:
    def __enter__(self):
        return None

    def __exit__(self, *exc):
        return False


# ---------------------------------------------------------------------
# Environment tweaks
# ---------------------------------------------------------------------
os.environ["OPENCV_IO_ENABLE_OPENEXR"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["ATTN_BACKEND"] = "xformers"

# Check if we are in a notebook
try:
    from trellis2.pipelines import Trellis2ImageTo3DPipeline
    from trellis2.modules.sparse import SparseTensor
except ImportError:
    print(
        "Error: trellis2 module not found. Make sure you are in the correct environment."
    )

from PIL import Image

os.environ["PYOPENGL_PLATFORM"] = (
    "egl"  # must come before importing pyrender, OpenGL, etc.
)

# --- pyrender imports must come after pyglet option set ---
import pyglet
import trimesh
import copy

pyglet.options["shadow_window"] = False

from pyrender import (
    PerspectiveCamera,
    DirectionalLight,
    SpotLight,
    PointLight,
    Mesh,
    Scene,
    Viewer,
    OffscreenRenderer,
)
import pyrender.constants as pyrc

In [ ]:
class Config:
    # Authentication
    hf_token: str = "YOUR_HUGGINGFACE_TOKEN_HERE"

    # Paths
    dataset_root: Path = Path("datasets/ObjaverseXL_sketchfab")
    out_dir: Path = Path(
        "datasets/ObjaverseXL_sketchfab/decode_shapes/vae_exp2"
    )  # Output directory

    # Model Settings
    model_id: str = "microsoft/TRELLIS.2-4B"
    low_vram: bool = True  # Enable low VRAM mode
    dtype: str = "fp32"  # Choices: "fp16", "bf16", "fp32"

    # Decoding Logic
    use_ss_decoder: bool = True  # Use sparse_structure_decoder for coords
    ss_target_res: int = 64  # Target resolution for SS decoder
    pool_on_cpu: bool = True  # Pool SS occupancy on CPU to save VRAM

    # Processing
    decode_resolutions: list[int] = [1024, 512, 256]
    decimate_faces: int = 16_777_216  # If >0, simplify mesh


args = Config()

# Authenticate
if args.hf_token and args.hf_token != "YOUR_HUGGINGFACE_TOKEN_HERE":
    login(token=args.hf_token)
else:
    print(
        "Warning: No HF token provided. Ensure you are logged in via CLI or provide token in Config."
    )

# Create output dir
args.out_dir.mkdir(parents=True, exist_ok=True)
print(f"Output directory set to: {args.out_dir}")

In [ ]:
# ---------------------------------------------------------------------
# Logging Setup
# ---------------------------------------------------------------------
# Reset handlers to avoid duplicate logs if cell is re-run
logging.root.handlers = []
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)],
)
LOGGER = logging.getLogger("decode_shapes")


def log_cuda_memory(tag: str) -> None:
    if not torch.cuda.is_available():
        return
    alloc = torch.cuda.memory_allocated() / (1024**3)
    reserved = torch.cuda.memory_reserved() / (1024**3)
    peak = torch.cuda.max_memory_allocated() / (1024**3)
    LOGGER.info(
        f"[{tag}] CUDA alloc={alloc:.2f}GB reserved={reserved:.2f}GB peak={peak:.2f}GB"
    )


def cleanup_cuda(*tensors) -> None:
    """Explicitly delete tensors and clear cache."""
    for t in tensors:
        try:
            del t
        except Exception:
            pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def add_batch_dim_if_missing(coords: torch.Tensor) -> torch.Tensor:
    # Expected SparseTensor coords are often [N,4] = [b,x,y,z]
    if coords.ndim != 2:
        raise ValueError(f"coords must be 2D, got {coords.shape}")
    if coords.shape[1] == 3:
        zeros = torch.zeros(
            (coords.shape[0], 1), dtype=torch.int32, device=coords.device
        )
        coords = torch.cat([zeros, coords.to(torch.int32)], dim=1)
    return coords.to(torch.int32).contiguous()


def load_shape_latent_npz(path: Path) -> Tuple[np.ndarray, np.ndarray]:
    with np.load(path) as data:
        feats = data["feats"]
        coords = data["coords"]
    return feats, coords


def load_ss_latent_npz(path: Path) -> np.ndarray:
    with np.load(path) as data:
        z = data["z"]
    return z


def generate_combinations(
    ss_dir: Path, shape_dir: Path
) -> Iterator[Tuple[str, Path, Path]]:
    ss_map: Dict[str, Path] = {p.stem: p for p in ss_dir.glob("*.npz")}
    shape_map: Dict[str, Path] = {p.stem: p for p in shape_dir.glob("*.npz")}

    with open('datasets/ObjaverseXL_sketchfab/vae_exps.json') as f:
        d = json.load(f)
        for item in d:
            name = item[0]
            ss_map_name = item[1]
            shape_map_name = item[2]
            if ss_map_name in ss_map and shape_map_name in shape_map:
                yield name, ss_map[ss_map_name], shape_map[shape_map_name], ss_map[
                    shape_map_name
                ], shape_map[ss_map_name]

In [ ]:
def coords_to_xyz(coords: torch.Tensor) -> np.ndarray:
    """
    coords: [N,4] int tensor with columns [b,x,y,z] or [N,3] [x,y,z]
    returns: [N,3] float numpy array [x,y,z]
    """
    if coords.shape[1] == 4:
        xyz = coords[:, 1:4]
    elif coords.shape[1] == 3:
        xyz = coords
    else:
        raise ValueError(f"coords must have shape [N,3] or [N,4], got {coords.shape}")
    return xyz.detach().cpu().numpy().astype(np.float32)


def plot_voxels_scatter(
    coords: torch.Tensor, voxel_size=1.0, origin=(0, 0, 0), max_points=200_000
):
    xyz = coords_to_xyz(coords)

    # Convert indices -> world coordinates (voxel centers)
    origin = np.array(origin, dtype=np.float32)
    xyz_world = origin + (xyz + 0.5) * float(voxel_size)

    # Downsample if huge
    if xyz_world.shape[0] > max_points:
        idx = np.random.choice(xyz_world.shape[0], size=max_points, replace=False)
        xyz_world = xyz_world[idx]

    fig = plt.figure()
    ax = fig.add_subplot(111, projection="3d")
    ax.scatter(xyz_world[:, 0], xyz_world[:, 1], xyz_world[:, 2], s=10)

    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    ax.set_zlabel("Z")

    # Rough equal aspect
    mins = xyz_world.min(axis=0)
    maxs = xyz_world.max(axis=0)
    spans = maxs - mins
    center = (mins + maxs) / 2
    radius = spans.max() / 2
    ax.set_xlim(center[0] - radius, center[0] + radius)
    ax.set_ylim(center[1] - radius, center[1] + radius)
    ax.set_zlim(center[2] - radius, center[2] + radius)

    plt.tight_layout()
    plt.show()


def plot_voxel_slice(coords: torch.Tensor, axis="z", index=None):
    """
    Shows a 2D occupancy slice (great for sanity checks).
    axis: 'x'|'y'|'z'
    index: slice coordinate; if None, uses median.
    """
    xyz = coords_to_xyz(coords).astype(np.int32)
    x, y, z = xyz[:, 0], xyz[:, 1], xyz[:, 2]

    if axis == "x":
        if index is None:
            index = int(np.median(x))
        mask = x == index
        a, b = y[mask], z[mask]
        a_name, b_name = "y", "z"
    elif axis == "y":
        if index is None:
            index = int(np.median(y))
        mask = y == index
        a, b = x[mask], z[mask]
        a_name, b_name = "x", "z"
    elif axis == "z":
        if index is None:
            index = int(np.median(z))
        mask = z == index
        a, b = x[mask], y[mask]
        a_name, b_name = "x", "y"
    else:
        raise ValueError("axis must be one of 'x','y','z'")

    if mask.sum() == 0:
        print(f"No voxels in {axis}={index}")
        return

    a_min, a_max = a.min(), a.max()
    b_min, b_max = b.min(), b.max()
    img = np.zeros((b_max - b_min + 1, a_max - a_min + 1), dtype=np.uint8)
    img[b - b_min, a - a_min] = 255

    plt.figure()
    plt.imshow(img, origin="lower", interpolation="nearest")
    plt.title(f"Occupancy slice {axis}={index} ({a_name} vs {b_name})")
    plt.xlabel(a_name)
    plt.ylabel(b_name)
    plt.show()


def nearest_neighbor_kdtree(
    coords_a: torch.Tensor, coords_b: torch.Tensor, batch_id: int = 0, k:int=1
):
    """
    Returns:
      nn_idx: (Na,) indices into B for nearest neighbor of each A
      nn_dist: (Na,) Euclidean distance
    """
    from scipy.spatial import cKDTree

    # Filter by batch if needed
    if coords_a.shape[1] == 4:
        coords_a = coords_a[coords_a[:, 0] == batch_id][:, 1:4]
    if coords_b.shape[1] == 4:
        coords_b = coords_b[coords_b[:, 0] == batch_id][:, 1:4]

    a = coords_a.detach().cpu().numpy().astype(np.float32)
    b = coords_b.detach().cpu().numpy().astype(np.float32)

    tree = cKDTree(b)
    nn_dist, nn_idx = tree.query(a, k=k, workers=-1)  # workers=-1 uses all cores

    return torch.from_numpy(nn_idx), torch.from_numpy(nn_dist)


In [ ]:
def decode_sparse_structure_coords(
    pipeline: Trellis2ImageTo3DPipeline,
    z_s: torch.Tensor,
    target_resolution: int = 64,
    pool_on_cpu: bool = True,
) -> torch.Tensor:
    """
    Decodes sparse structure latent z_s into coordinates.
    """
    decoder = pipeline.models["sparse_structure_decoder"]

    if pipeline.low_vram:
        decoder.to(pipeline.device)

    decoded = decoder(z_s) > 0  # bool tensor

    if pipeline.low_vram:
        decoder.cpu()

    LOGGER.info(f"Decoded sparse structure shape before max pool: {decoded.shape}")

    if decoded.shape[2] != target_resolution:
        ratio = decoded.shape[2] // target_resolution
        if pool_on_cpu:
            decoded_cpu = decoded.to("cpu")
            pooled = F.max_pool3d(decoded_cpu.to(torch.float16), ratio, ratio, 0) > 0.5
            decoded = pooled  # stays on CPU
        else:
            decoded = F.max_pool3d(decoded.to(torch.float16), ratio, ratio, 0) > 0.5

    LOGGER.info(f"Decoded sparse structure shape after max pool: {decoded.shape}")

    coords = (
        decoded.nonzero(as_tuple=False)[:, [0, 2, 3, 4]].to(torch.int32).contiguous()
    )
    return coords, decoded


def decode_meshes_from_shape_slat(
    pipeline: Trellis2ImageTo3DPipeline,
    shape_slat: SparseTensor,
    resolution: int,
    return_subs: bool = False,
):
    """
    Decodes shape features into a mesh.
    """
    shape_decoder = pipeline.models["shape_slat_decoder"]
    shape_decoder.set_resolution(resolution)

    if pipeline.low_vram:
        shape_decoder.to(pipeline.device)
        shape_decoder.low_vram = True

    ret = shape_decoder(shape_slat, return_subs=return_subs)

    if pipeline.low_vram:
        shape_decoder.cpu()
        shape_decoder.low_vram = False

    if return_subs:
        meshes, subs = ret
        return meshes, subs

    if isinstance(ret, tuple) and len(ret) == 2:
        meshes, _subs = ret
        return meshes

    return ret

In [ ]:
def export_voxels_as_cubes_mesh(
    out_path: str,
    coords: torch.Tensor,
    voxel_size=1.0,
    origin=(0, 0, 0),
    max_voxels=500_000,
):
    import open3d as o3d

    if coords.shape[1] == 4:
        xyz = coords[:, 1:4]
    else:
        xyz = coords

    xyz = xyz.detach().cpu().numpy().astype(np.float32)
    origin = np.array(origin, dtype=np.float32)

    if xyz.shape[0] > max_voxels:
        idx = np.random.choice(xyz.shape[0], size=max_voxels, replace=False)
        xyz = xyz[idx]
        print(f"Downsampled to {max_voxels} voxels for cube-mesh export.")

    mesh_all = o3d.geometry.TriangleMesh()
    for v in xyz:
        # cube corner at voxel origin (not center)
        corner = origin + v * float(voxel_size)
        cube = o3d.geometry.TriangleMesh.create_box(
            width=voxel_size, height=voxel_size, depth=voxel_size
        )
        cube.translate(corner)
        mesh_all += cube

    mesh_all.compute_vertex_normals()
    o3d.io.write_triangle_mesh(out_path, mesh_all)
    print(f"Wrote cube-mesh: {out_path}")



def write_ply_binary(path: Path, vertices: np.ndarray, faces: np.ndarray) -> None:
    """
    Writes a binary_little_endian PLY with vertex positions and triangular faces.
    """
    path.parent.mkdir(parents=True, exist_ok=True)

    v = np.asarray(vertices, dtype=np.float32)
    f = np.asarray(faces, dtype=np.int32)

    with path.open("wb") as f_out:
        header = (
            "ply\n"
            "format binary_little_endian 1.0\n"
            f"element vertex {v.shape[0]}\n"
            "property float x\n"
            "property float y\n"
            "property float z\n"
            f"element face {f.shape[0]}\n"
            "property list int int vertex_indices\n"
            "end_header\n"
        )
        f_out.write(header.encode("ascii"))
        f_out.write(v.tobytes())

        counts = np.full((f.shape[0], 1), 3, dtype=np.int32)
        faces_data = np.hstack([counts, f])
        f_out.write(faces_data.tobytes())

In [ ]:
# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type != "cuda":
    raise RuntimeError("This script expects a CUDA GPU.")

dtype_map = {"fp16": torch.float16, "bf16": torch.bfloat16, "fp32": torch.float32}
compute_dtype = dtype_map[args.dtype]

LOGGER.info(f"Loading pipeline {args.model_id} ...")

# Load Pipeline
pipeline = Trellis2ImageTo3DPipeline.from_pretrained(args.model_id)
pipeline.cuda()
pipeline.low_vram = bool(args.low_vram)

LOGGER.info(f"Pipeline loaded. low_vram={pipeline.low_vram}")
torch.cuda.reset_peak_memory_stats()
log_cuda_memory("after_pipeline_load")

In [ ]:
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

In [ ]:
# Create context for mixed precision
autocast_ctx = (
    torch.autocast(device_type="cuda", dtype=compute_dtype)
    if compute_dtype != torch.float32
    else nullcontext()
)

# Run Inference
with torch.inference_mode(), autocast_ctx:
    for res in [1024]:
        # Construct paths based on resolution
        ss_dir = args.dataset_root / "ss_latents" / f"ss_enc_conv3d_16l8_fp16_64_{res}"
        shape_dir = (
            args.dataset_root / "shape_latents" / f"shape_enc_next_dc_f16c32_fp16_{res}"
        )

        if not ss_dir.exists() or not shape_dir.exists():
            LOGGER.warning(f"Missing dirs for res={res}: \n  {ss_dir} \n  {shape_dir}")
            continue

        LOGGER.info(
            f"Decoding resolution={res} from:\n  ss: {ss_dir}\n  shape: {shape_dir}"
        )

        # Process files
        exp_combinations = generate_combinations(ss_dir, shape_dir)
        key, ss_path, shape_path, ss_path_temp, shape_path_temp = list(exp_combinations)[-1]
        torch.cuda.reset_peak_memory_stats()

        LOGGER.info(f"Processing: {key}")

        # 1. Load latents (CPU)
        feats_np, coords_np = load_shape_latent_npz(shape_path)
        z_np = load_ss_latent_npz(ss_path) if args.use_ss_decoder else None
        feats_np_temp, _ = load_shape_latent_npz(shape_path_temp)

        # 2. Move to GPU
        feats = torch.from_numpy(feats_np).to(
            device=device, dtype=compute_dtype, non_blocking=True
        )

        enc_coords = torch.from_numpy(coords_np).to(
            device=device, dtype=torch.int32, non_blocking=True
        )
        enc_coords = add_batch_dim_if_missing(enc_coords)

        LOGGER.info(f"Loaded latents: feats shape {feats.shape}, coords shape {enc_coords.shape}")

        feats_temp = torch.from_numpy(feats_np_temp).to(
            device=device, dtype=compute_dtype, non_blocking=True
        )

        # 3. Optional: Decode Sparse Structure coords
        if args.use_ss_decoder:
            LOGGER.info("Decoding sparse structure coordinates...")
            z_s = torch.from_numpy(z_np).to(
                device=device, dtype=compute_dtype, non_blocking=True
            )
            LOGGER.info(
                f"Loaded latents: z_s shape {z_s.shape} "
            )
            if z_s.ndim == 4:
                z_s = z_s.unsqueeze(0)

            dec_coords, decoded = decode_sparse_structure_coords(
                pipeline,
                z_s,
                target_resolution=64,
                pool_on_cpu=args.pool_on_cpu,
            )
            dec_coords = dec_coords.to(device)

        LOGGER.info(f"Decoded latents coords shape {dec_coords.shape}")

        # take intersection of dec_coords and enc_coords
        dec_coords_set = set(map(tuple, dec_coords.cpu().numpy()))
        enc_coords_set = set(map(tuple, enc_coords.cpu().numpy()))
        coords_torch = (
            torch.from_numpy(np.array(list(dec_coords_set & enc_coords_set)))
            .to(torch.int32)
            .contiguous()
            .to(device)
        )
        # update feats to match coords
        coord_to_index = {tuple(c.cpu().numpy()): i for i, c in enumerate(dec_coords)}
        indices = [coord_to_index[tuple(c.cpu().numpy())] for c in coords_torch]
        feats_temp = feats_temp[indices]
        #feats_temp =  feats[indices]  # feats_temp

        for k in [10, 50, 100, 250, 500, 1500, 3000]:
            nn_idx, nn_dist = nearest_neighbor_kdtree(coords_torch, coords_torch, k=k)
            coords = coords_torch[nn_idx[:1]][0]
            feats = feats_temp[nn_idx[:1]][0]

            # ## NN features
            # alpha = 0.0
            # nn_idx, nn_dist = nearest_neighbor_kdtree(dec_coords, enc_coords)
            # feats = feats[nn_idx] * alpha + feats_temp * (1.0 - alpha)
            # coords = dec_coords

            LOGGER.info(
                f"After intersection, feats shape: {feats.shape}, coords shape: {coords.shape}"
            )

            export_voxels_as_cubes_mesh(
                args.out_dir
                / f"VOXELS_{key}_{res}_{k}{'_sscoords.ply' if args.use_ss_decoder else '.ply'}",
                coords,
                voxel_size=1.0,
            )

            # 4. Construct SparseTensor
            shape_slat = SparseTensor(feats=feats, coords=coords)
            LOGGER.info(
                f"Constructing SparseTensor with feats {feats.shape} and coords {coords.shape}"
            )

            log_cuda_memory(f"{key}: before_decode")

            # 5. Decode Meshes
            meshes = decode_meshes_from_shape_slat(
                pipeline,
                shape_slat,
                resolution=res,
                return_subs=False,
            )

            # 6. Post-processing
            mesh = meshes[0]
            # mesh.fill_holes()

            # if args.decimate_faces and args.decimate_faces > 0:
            #     mesh.simplify(args.decimate_faces)

            # 7. Export
            v = mesh.vertices.detach().cpu().numpy().astype(np.float32)
            f = mesh.faces.detach().cpu().numpy().astype(np.int32)

            out_name = f"{key}_{res}_{k}" + ("_sscoords" if args.use_ss_decoder else "")
            out_path = args.out_dir / f"{out_name}.ply"
            write_ply_binary(out_path, v, f)

        log_cuda_memory(f"{key}: after_export")
        LOGGER.info(f"Wrote {out_path}")

        # 8. Cleanup to prevent VRAM accumulation
        cleanup_cuda(shape_slat, feats, coords, meshes, mesh)
        if args.use_ss_decoder:
            cleanup_cuda(z_s)

LOGGER.info("Done.")

## Visualizations

In [ ]:
# viz_mesh = trimesh.Trimesh(vertices=v, faces=f)
# viz_mesh.apply_scale(0.6)
# center = viz_mesh.bounds.mean(axis=0)  # (min+max)/2
# viz_mesh.apply_translation(-center)
# # mesh.rezero()
# # mesh.fix_normals()
# viz_mesh = Mesh.from_trimesh(viz_mesh, smooth=True)

In [ ]:
# base_pose = np.array(
#     [
#         [1.0, 0.0, 0.0, 0.0],
#         [0.0, 0.0, -1.0, 0.0],
#         [0.0, 1.0, 0.0, -0.1],
#         [0.0, 0.0, 0.0, 1.0],
#     ]
# )
# direc_l = DirectionalLight(color=np.ones(3), intensity=1.0)
# spot_l = SpotLight(
#     color=np.ones(3),
#     intensity=1.0,
#     innerConeAngle=np.pi / 16,
#     outerConeAngle=np.pi / 6,
# )
# cam = PerspectiveCamera(yfov=(np.pi / 4.0))
# cam_pose = np.array(
#     [
#         #  right_x  up_x   localZ_x   cam_x
#         [0.0, 0.0, 1.0, 0.8],
#         #  right_y  up_y   localZ_y   cam_y
#         [1.0, 0.0, 0.0, 0.0],
#         #  right_z  up_z   localZ_z   cam_z
#         [0.0, 1.0, 0.0, -0.1],
#         [0.0, 0.0, 0.0, 1.0],
#     ]
# )
# scene = Scene(
#     ambient_light=np.array([0.8, 0.3, 0.88, 1.0]), bg_color=[1.0, 1.0, 1.0, 1.0]
# )
# node = scene.add(viz_mesh, pose=base_pose)
# _ = scene.add(direc_l, pose=cam_pose)
# # _ = scene.add(spot_l, pose=cam_pose)
# cam_node = scene.add(cam, pose=cam_pose)
# # ==============================================================================
# # Offscreen renderer
# # ==============================================================================
# width, height = 512 * 2, 512 * 2
# r = OffscreenRenderer(viewport_width=width, viewport_height=height)
# # ==============================================================================
# # Turntable loop
# # ==============================================================================
# # Number of frames around the full 360° (you can increase for smoother animation)
# num_frames = 3
# # Compute object center in world coords (here we use the translation part of base_pose)
# object_center = base_pose[:3, 3]
# axis_vector = [0, 0.6, 1]
# combined = []
# for i, angle in enumerate(np.linspace(0, 2 * np.pi, num_frames, endpoint=False)):
#     # Build a rotation matrix around Y axis centered on object_center
#     R = trimesh.transformations.rotation_matrix(
#         angle, axis_vector, point=object_center
#     )
#     # Apply rotation to the base pose
#     new_pose = R.dot(base_pose)
#     # Update the node’s transform
#     node.matrix = new_pose
#     # Render
#     color, depth = r.render(scene, flags=pyrc.RenderFlags.SKIP_CULL_FACES)  #
#     combined.append(color)
# combined = np.concatenate(combined, axis=1)
# Image.fromarray(combined, mode="RGB")
# r.delete()

In [ ]:
# plt.figure(figsize=(20, 12))
# plt.imshow(combined)
# plt.axis("off")
# plt.show()

In [ ]:
# plot_voxels_scatter(coords, voxel_size=1, origin=(0, 0, 0))
# plot_voxel_slice(coords, axis="z", index=None)

In [ ]:
# from matplotlib import animation
# from IPython.display import HTML

# RESHAPE = False
# if RESHAPE:
#     ratio = decoded.shape[2] // 16
#     decoded = decoded.to("cpu")
#     pooled = F.max_pool3d(decoded.to(torch.float16), ratio, ratio, 0) > 0.5
#     cube1 = pooled[0, 0].numpy().astype(bool)
# else:
#     cube1 = decoded[0, 0].cpu().numpy().astype(bool)

# # combine objects (already boolean)
# voxelarray = cube1

# # set colors
# colors = np.empty(voxelarray.shape, dtype=object)
# colors[voxelarray] = "blue"

# # --- build figure ---
# fig = plt.figure(figsize=(8, 8))
# ax = fig.add_subplot(111, projection="3d")

# # draw voxels once
# vox = ax.voxels(voxelarray, facecolors=colors, edgecolor="k")

# # nicer view limits/aspect (optional)
# ax.set_xlim(0, voxelarray.shape[0])
# ax.set_ylim(0, voxelarray.shape[1])
# ax.set_zlim(0, voxelarray.shape[2])
# ax.grid(False)
# ax.set_xticks([])
# ax.set_yticks([])
# ax.set_zticks([])


# # rotate camera
# def update(frame):
#     ax.view_init(elev=10, azim=frame)  # azim rotates around
#     return []


# anim = animation.FuncAnimation(
#     fig,
#     update,
#     frames=np.linspace(0, 360, 12),  # 120 frames for a full turn
#     interval=50,  # ms between frames
#     blit=False,
# )

# plt.close(fig)  # prevents a static figure from showing above the animation
# # HTML(anim.to_jshtml())
# anim.save(
#     args.out_dir / f"{key}_{res}{'_sscoords.mp4' if args.use_ss_decoder else '.mp4'}",
#     writer="ffmpeg",
#     fps=2,
#     dpi=150,
# )